# Optimize squeezing by changing one input

Vary only the **peak power of the input Gaussian pulse**, starting at 100 kW.
Keep the 200 fs pulse width, LP01 launch, three-mode fiber model, 2.86 m length,
and spectral filter from `step_index_squeezing.ipynb` fixed.

We minimize the detected photon-number Fano factor in decibels:

\[
F(P)=\frac{\operatorname{Var}[N_{\mathrm{filtered}}]}{\langle N_{\mathrm{filtered}}\rangle},
\qquad J(P)=10\log_{10}F(P).
\]

**More negative is better; 0 dB is shot noise.** Run the cells in order with
**GNLSE (Julia)**. This is a local optimization demo, not a
search for the global optimum. Changing power also changes pulse energy.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = joinpath(workspace, "FastDifferentiableMMNLO")
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "SQUEEZING_RUN_DIR") ? ENV["SQUEEZING_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Total launch power", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_step_index_power_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "step_index_power_optimization.jl"))

## 1. Choose the control and the compute budget

Use dimensionless power \(u=P/(100\,\mathrm{kW})\), so the learning rate has a
useful numerical scale. Power is bounded between 80 and 120 kW. Require at
least 50% of the baseline detected photon number. These are editable demo
constraints, not experimentally established limits.

The search uses a 10 mm propagation step. Final results are checked at 1 mm.
The setup check below also compares two finite-difference step sizes.

In [ ]:
reference_power_W = 100_000.0
power_bounds_W = (80_000.0, 120_000.0)
minimum_brightness_fraction = 0.50
fd_step = 0.002                  # ±200 W when differentiating with respect to u
learning_rate = 0.001
max_iterations = 12
max_optimizer_evaluations = 50
gradient_tolerance = 0.05        # dB per unit u
max_power_change_W = 10_000.0
search_dz = 0.01                # m; shared by forward and discrete adjoint
validation_dz = 0.001           # m

settings = Dict(
    "control" => "input Gaussian peak power (W)",
    "objective" => "10log10(filtered photon-number Fano factor)",
    "reference_power_W" => reference_power_W,
    "power_bounds_W" => collect(power_bounds_W),
    "minimum_brightness_fraction" => minimum_brightness_fraction,
    "finite_difference_step_u" => fd_step,
    "learning_rate" => learning_rate,
    "max_iterations" => max_iterations,
    "max_optimizer_evaluations" => max_optimizer_evaluations,
    "gradient_tolerance" => gradient_tolerance,
    "max_power_change_W" => max_power_change_W,
    "search_dz_m" => search_dz,
    "validation_dz_m" => validation_dz,
    "Nt" => 2048, "time_window_ps" => 10.0, "pulse_fwhm_ps" => 0.2,
    "fiber_length_m" => 2.86, "carrier_m" => 1379.2e-9,
    "core_radius_m" => 52.5e-6, "NA" => 0.1, "num_modes" => 3,
    "filter_requested_cutoff_m" => 1370.6998e-9,
    "cp_target_error" => 1e-3, "cp_seed" => Int(0x4d4d474e),
    "adjoint" => "discrete_rk4ip", "checkpoint_stride" => 32,
    "noise_model" => "linearized input vacuum noise; no distributed Raman or detector noise",
    "julia_version" => string(VERSION),
    "manifest_sha256" => bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
open(joinpath(run_dir, "settings.toml"), "w") do io
    TOML.print(io, settings)
end

## 2. Build the fixed fiber and filter once

CP compression is computed once and reused at every power. This makes the
objective deterministic and avoids recompressing the nonlinear coupling tensor
inside the optimization loop.

In [ ]:
time_grid = TimeGrid(settings["Nt"], settings["time_window_ps"])
domain = MMGNLSEDomain(degrees_of_freedom(:time, :space), time_grid)
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
parameters_cp = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=properties.n2, omega0=properties.omega0, raman=properties.raman)

wavelengths = wavelength_axis(time_grid, properties.omega0)
cutoff_index = argmin(abs.(wavelengths .- settings["filter_requested_cutoff_m"]))
transmission = zeros(length(wavelengths))
transmission[1:cutoff_index] .= 1.0
photon_observable = SpectralPhotonNumber(filter=transmission, modes=:all)
all_photons = SpectralPhotonNumber(filter=ones(length(wavelengths)), modes=:all)
@printf("Actual filter edge: %.6f nm; CP relative error: %.3g\n",
    wavelengths[cutoff_index]*1e9, S_cp.relative_error)

## 3. Turn the squeezing calculation into a function

Each objective evaluation launches the pulse, solves the MMGNLSE, computes the
filtered mean photon number, and propagates its adjoint. The squared norm of
the photon-normalized input sensitivity gives the linearized input-vacuum
variance, as in the original notebook.

Here the **discrete RK4IP adjoint** differentiates the same numerical propagation
map as the forward solver. Its forward and adjoint step sizes match. The original
notebook used a continuous adjoint, so small numerical differences are expected.

This is the original noise model: deterministic Kerr/Raman propagation with
input vacuum fluctuations. It omits distributed Raman Langevin noise, loss noise,
and detector noise; the result is a model prediction, not a complete experimental
squeezing prediction. No random ensemble is sampled inside the optimizer.

In [ ]:
evaluation_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
evaluations = NamedTuple[]

## 4. Check the starting point and the numerical gradient

The outer derivative is a central finite difference of the **entire Fano-factor
objective**, including its power-dependent denominator:

\[
g(u)\approx\frac{J(u+h)-J(u-h)}{2h}.
\]

The photon-number adjoint alone is not the derivative of squeezing. An analytic
outer gradient would need derivatives of that adjoint calculation, i.e. higher
order derivatives. Central differences are a transparent first choice for one
control. Halving \(h\) checks its sensitivity to the numerical difference step.

In [ ]:
baseline = squeezing_at(reference_power_W)
baseline_fine = squeezing_at(reference_power_W; dz=validation_dz)
g_start = power_gradient(1.0)
g_start_half = power_gradient(1.0; h=fd_step/2)
gradient_relative_change = abs(g_start-g_start_half)/max(abs(g_start_half), 1e-8)
baseline_step_error_dB = abs(baseline.squeezing_dB-baseline_fine.squeezing_dB)
# Check dz whether too big!
@assert baseline_step_error_dB < 0.02 "Reduce search_dz before optimization."
# Check whether the descent step is too big, making gradient inaccurate. Assert that the relative change in the gradient is small and that the difference between the baseline and fine baseline is within tolerance.
@assert gradient_relative_change < 0.01 "Refine the finite-difference step before optimization."
brightness_floor = minimum_brightness_fraction * baseline_fine.photons
@printf("Starting gradient: %.6f dB/u; half-step: %.6f dB/u (%.3f%% change)\n",
    g_start, g_start_half, 100gradient_relative_change)
@printf("10 mm versus 1 mm starting objective difference: %.6f dB\n", baseline_step_error_dB)

## 5. Projected gradient descent with backtracking

Propose \(u_{\mathrm{new}}=\operatorname{clip}(u-\eta g,u_{\min},u_{\max})\).
If the trial loses brightness or fails to decrease the objective sufficiently,
halve the step and try again. Only accepted steps enter the history. Stop on a
small projected gradient, failed line search, or the stated iteration/evaluation
budget. The displayed status distinguishes these outcomes.

In [ ]:
optimization = optimize_power()
println("Stop: ", optimization.status)
@printf("Best accepted power: %.4f kW; squeezing: %.6f dB\n",
    optimization.best.power_W/1000, optimization.best.squeezing_dB)

## 6. Validate the result

Recompute the baseline and selected power with a 1 mm step. Also check the
selected point at 5 mm and test nearby powers at that resolution. These checks
establish local improvement and step-size stability within the fixed three-mode,
2048-sample model; they do not establish convergence with respect to the number
of spatial modes or certify a global optimum.

In [ ]:
best_power_W = optimization.best.power_W
best_fine = squeezing_at(best_power_W; dz=validation_dz)
best_mid = squeezing_at(best_power_W; dz=search_dz/2)
neighbor_delta_W = 500.0
left_neighbor = squeezing_at(max(power_bounds_W[1], best_power_W-neighbor_delta_W); dz=search_dz/2)
right_neighbor = squeezing_at(min(power_bounds_W[2], best_power_W+neighbor_delta_W); dz=search_dz/2)

improvement_dB = baseline_fine.squeezing_dB-best_fine.squeezing_dB
best_step_error_dB = abs(optimization.best.squeezing_dB-best_fine.squeezing_dB)
@assert improvement_dB > 0 "No validated improvement over the starting point."
@assert best_fine.photons >= brightness_floor "Brightness constraint fails on the finer grid."
@assert best_step_error_dB < 0.02 "Refine search_dz and rerun optimization."
@assert best_mid.squeezing_dB <= min(left_neighbor.squeezing_dB, right_neighbor.squeezing_dB)
@assert all(diff([r.squeezing_dB for r in optimization.history]) .< 0)

summary = Dict(
    "status"=>optimization.status,
    "baseline_power_W"=>reference_power_W, "best_power_W"=>best_power_W,
    "baseline_squeezing_dB"=>baseline_fine.squeezing_dB,
    "best_squeezing_dB"=>best_fine.squeezing_dB,
    "baseline_fano"=>baseline_fine.fano, "best_fano"=>best_fine.fano,
    "improvement_dB"=>improvement_dB,
    "relative_fano_reduction"=>1-best_fine.fano/baseline_fine.fano,
    "baseline_photons"=>baseline_fine.photons, "best_photons"=>best_fine.photons,
    "brightness_ratio"=>best_fine.photons/baseline_fine.photons,
    "best_photon_transmission"=>best_fine.photon_transmission,
    "baseline_step_error_dB"=>baseline_step_error_dB,
    "best_step_error_dB"=>best_step_error_dB,
    "gradient_step_relative_change"=>gradient_relative_change,
    "accepted_steps"=>length(optimization.history)-1,
    "optimization_seconds"=>optimization.seconds,
    "new_optimizer_objective_evaluations"=>optimization.new_objective_evaluations,
    "cp_relative_error"=>S_cp.relative_error,
    "actual_filter_edge_nm"=>wavelengths[cutoff_index]*1e9,
    "validation_passed"=>true,
)
open(joinpath(run_dir, "summary.toml"), "w") do io
    TOML.print(io, summary)
end
@printf("Validated: %.3f → %.3f kW; %.5f → %.5f dB\n",
    reference_power_W/1000, best_power_W/1000,
    baseline_fine.squeezing_dB, best_fine.squeezing_dB)
@printf("Fano reduction: %.2f%%; detected brightness retained: %.2f%%\n",
    100summary["relative_fano_reduction"], 100summary["brightness_ratio"])
@printf("Optimization: %.1f s, %d accepted steps, %d new objective evaluations\n",
    optimization.seconds, summary["accepted_steps"], optimization.new_objective_evaluations)

## 7. Look at the power dependence and optimization path

The local sweep below is for visualization; it is evaluated **after** gradient
descent and does not select the optimized power. Sweep points use the search
step; the validated baseline and final point use the finer step.

In [ ]:
scan_lower = max(power_bounds_W[1], min(best_power_W, reference_power_W)-4_000)
scan_upper = min(power_bounds_W[2], max(best_power_W, reference_power_W)+4_000)
scan = [squeezing_at(P) for P in range(scan_lower, scan_upper; length=11)]
save_rows("power_sweep.csv", scan)

fig, axes = subplots(1, 2; figsize=(9, 3.5), constrained_layout=true)
axes[1].plot([r.power_W/1000 for r in scan], [r.squeezing_dB for r in scan],
    "o-"; color="#366b9e", label="Local power sweep")
axes[1].plot([r.power_W/1000 for r in optimization.history],
    [r.squeezing_dB for r in optimization.history], "s--";
    color="#cf7c32", label="Accepted gradient steps", markersize=4)
axes[1].scatter([reference_power_W/1000, best_power_W/1000],
    [baseline_fine.squeezing_dB, best_fine.squeezing_dB];
    color="black", marker="x", s=60, label="1 mm validation", zorder=5)
axes[1].set_xlabel("Input peak power (kW)")
axes[1].set_ylabel("Photon-number noise (dB relative to shot noise)")
axes[1].set_title("One input: peak power")
axes[1].legend(; fontsize=8)
axes[2].plot([r.iteration for r in optimization.history],
    [r.squeezing_dB for r in optimization.history], "o-"; color="#366b9e")
axes[2].set_xlabel("Accepted iteration")
axes[2].set_ylabel("Photon-number noise (dB)")
axes[2].set_title("Gradient descent: lower is better")
for ax in axes
    ax.grid(; alpha=0.2)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
end
savefig(joinpath(run_dir, "power_optimization.png"); dpi=180)
savefig(joinpath(run_dir, "power_optimization.pdf"))
display(fig)
close(fig)
@printf("Total objective evaluations (setup, search, validation, plot): %d\n", length(evaluations))
@printf("Total measured objective time: %.1f s\n", sum(r.seconds for r in evaluations))